# 02 — Generate paper briefs from the corpus

Read frozen full text from `data/paper_brief_evaluation/corpus/` and write a new `{run_id}/` folder with generated briefs.

**Prerequisite:** notebook 01 has written `corpus/manifest.jsonl` and the matching `.txt` files. Set **MODEL** (required) before you run the generate cell. Optional **LIMIT**: leave blank (`None`) to generate every manifest row; a positive integer generates only the first N rows in file order. This notebook does **not** query Postgres and does **not** read or write `PaperBrief`. Run it with `just notebooks` (needs `OPENAI_*`). Do not use `just sandbox`.

Domain calls: `generate_paper_brief_content` and `load_paper_brief_template` from `paper_reviewer.topic_scope.generate_paper_brief.llm`. Do not import `paper_reviewer.flows` and do not call `create_paper_brief`.

**Git:** `{run_id}/` results under `data/paper_brief_evaluation/` are tracked so you can commit them. They stay out of the production image (`.dockerignore`).

In [1]:
# Chat model id for this generate run (required). Example: "llama3.1:8b" or "gemma4:e4b"
MODEL = "gemma4:e4b"

In [2]:
# Max papers to process in this run. Leave blank (None) to process every row.
# A positive integer takes the first N rows in file order and stops.
LIMIT = None

In [3]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path

from paper_reviewer.schemas.topic_scope.generate_paper_brief import PaperBriefLlmResult
from paper_reviewer.topic_scope.generate_paper_brief.llm import (
    generate_paper_brief_content,
    load_paper_brief_template,
)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
RUNS_PARENT = REPO_ROOT / "data" / "paper_brief_evaluation"
CORPUS_DIR = RUNS_PARENT / "corpus"
MANIFEST_PATH = CORPUS_DIR / "manifest.jsonl"
print(f"repo root: {REPO_ROOT}")
print(f"corpus dir: {CORPUS_DIR}")
print(f"manifest: {MANIFEST_PATH}")

repo root: /workspace
corpus dir: /workspace/data/paper_brief_evaluation/corpus
manifest: /workspace/data/paper_brief_evaluation/corpus/manifest.jsonl


In [4]:
def model_slug(model: str) -> str:
    slug = model.strip()
    for char in (":", "/", "\\", " "):
        slug = slug.replace(char, "-")
    return slug


def new_run_id(model: str) -> str:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{stamp}_{model_slug(model)}"


def load_manifest(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if stripped:
                rows.append(json.loads(stripped))
    return rows


def brief_success_record(doi: str, result: PaperBriefLlmResult) -> dict:
    return {
        "doi": doi,
        "brief": result.content.model_dump(mode="json"),
        "prompt_tokens": result.prompt_tokens,
        "completion_tokens": result.completion_tokens,
        "total_tokens": result.total_tokens,
    }


def brief_error_record(doi: str, error: str) -> dict:
    return {"doi": doi, "error": error}


def parse_limit(value: object) -> int | None:
    if value is None:
        return None
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return None
        if stripped.isdigit():
            n = int(stripped)
        else:
            raise RuntimeError(
                "LIMIT must be a positive integer, or left blank "
                "to process all. No run folder was created."
            )
    elif isinstance(value, bool) or not isinstance(value, int):
        raise RuntimeError(
            "LIMIT must be a positive integer, or left blank "
            "to process all. No run folder was created."
        )
    else:
        n = value
    if n < 1:
        raise RuntimeError(
            "LIMIT must be a positive integer, or left blank "
            "to process all. No run folder was created."
        )
    return n

In [5]:
model = MODEL.strip()
if not model:
    raise RuntimeError(
        "MODEL is required. Set the chat model id in the MODEL cell "
        '(example: "llama3.1:8b"). No run folder was created.'
    )

if not MANIFEST_PATH.is_file():
    raise RuntimeError(
        f"Missing corpus manifest: {MANIFEST_PATH}. "
        "Run notebook 01 (build corpus) first. No run folder was created."
    )

limit = parse_limit(LIMIT)
os.environ["OPENAI_MODEL"] = model
manifest_rows = load_manifest(MANIFEST_PATH)
total_rows = len(manifest_rows)
if limit is not None:
    manifest_rows = manifest_rows[:limit]
run_id = new_run_id(model)
run_dir = RUNS_PARENT / run_id
run_dir.mkdir(parents=True, exist_ok=False)
template_path = run_dir / "02-brief-template.md"
briefs_path = run_dir / "02-briefs.jsonl"
template_path.write_text(load_paper_brief_template(), encoding="utf-8")
briefs_path.write_text("", encoding="utf-8")
print(f"model: {model}")
print(f"run dir: {run_dir}")
if limit is None:
    print(f"limit: all ({total_rows})")
else:
    print(f"limit: {limit} of {total_rows}")
print(f"manifest rows: {len(manifest_rows)}")

accepted = 0
errors: list[tuple[str, str]] = []

for row in manifest_rows:
    doi = str(row.get("doi") or "(missing doi)")
    try:
        filename = row.get("filename")
        if not filename:
            raise ValueError("manifest row has no filename")
        title = row.get("title")
        if not title:
            raise ValueError("manifest row has no title")
        text_path = CORPUS_DIR / filename
        if not text_path.is_file():
            raise FileNotFoundError(f"missing corpus file: {text_path}")
        full_text = text_path.read_text(encoding="utf-8")
        result = generate_paper_brief_content(
            full_text,
            title=title,
            journal=row.get("journal"),
            published_year=row.get("published_year"),
        )
        record = brief_success_record(doi, result)
        accepted += 1
        print(f"OK {doi}")
    except Exception as exc:
        message = str(exc)
        record = brief_error_record(doi, message)
        errors.append((doi, message))
        print(f"ERROR {doi}: {exc}")
    with briefs_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("---")
print(f"accepted: {accepted}")
print(f"errors: {len(errors)}")
print(f"briefs: {briefs_path}")
print(f"template: {template_path}")

model: gemma4:e4b
run dir: /workspace/data/paper_brief_evaluation/20260818T221210Z_gemma4-e4b
limit: all (109)
manifest rows: 109
OK 10.1002/ECE3.74085
OK 10.1002/PROT.70113
OK 10.1002/RMV.70176
OK 10.1002/VMS3.71125
OK 10.1016/J.APSB.2026.02.021
OK 10.1016/J.CRPVBD.2026.100416
OK 10.1016/J.EKIR.2026.106544
OK 10.1016/J.GORE.2026.102155
OK 10.1016/J.IMMUNI.2026.05.013
OK 10.1016/J.ISCI.2026.115754
OK 10.1016/J.JBC.2026.113118
OK 10.1016/J.ONEHLT.2025.101310
ERROR 10.1016/J.ONEHLT.2026.101368: LLM returned no JSON object

Assistant output:
{
  "summary": "This study characterizes the molecular co-circulation of endemic and recently introduced West Nile Virus Lineage 1 clades following a major WNV outbreak in Andalusia, Spain, emphasizing the importance of advanced genomic surveillance.",
  "objective": "To provide a detailed molecular characterization of the last WNV Lineage 1 outbreak in Andalusia by documenting the coexistence of imported viruses alongside endemic viruses present in S

After a successful run, commit the `{run_id}/` folder if you want the briefs in the repository:

```bash
git add data/paper_brief_evaluation/
```

Production images still exclude `data/` via `.dockerignore`. Next: [03-evaluate-briefs.ipynb](03-evaluate-briefs.ipynb).